# Funnel Analysis to Calculate Step-by-step Drop-off Rates

In [4]:
# Page navigation logs by user（dummy data）
# Contains data where the same user has viewed the same page multiple times (duplicate data)
funnel_logs = [
    {"user_id": "U001", "step": "top_page"},
    {"user_id": "U001", "step": "product_page"},
    {"user_id": "U001", "step": "cart"},
    {"user_id": "U001", "step": "purchase"},
    {"user_id": "U002", "step": "top_page"},
    {"user_id": "U002", "step": "product_page"},
    {"user_id": "U002", "step": "top_page"}, # Duplicate access from U002
    {"user_id": "U003", "step": "top_page"},
    {"user_id": "U004", "step": "top_page"},
    {"user_id": "U004", "step": "product_page"},
    {"user_id": "U004", "step": "cart"},
    {"user_id": "U005", "step": "top_page"},
    {"user_id": "U005", "step": "product_page"},
    {"user_id": "U005", "step": "product_page"}, # Duplicate access from U005
]

# The correct order of the funnel
FUNNEL_STEPS = ["top_page", "product_page", "cart", "purchase"]

def get_users_at_step(logs, step):
    """
    Args:
        logs (list): Funnel logs. Each log is a dictionary containing user_id and step
        step (str): The name of the funnel step to filter by
    Return:
        set: A set of unique user IDs who reached the given step
    """
    users = set()
    for log in logs:
        if log["step"] == step:
            users.add(log["user_id"])
    return users

step_counts = {}
for step in FUNNEL_STEPS:
    step_counts[step] = len(get_users_at_step(funnel_logs, step))


def analyze_funnel(logs, steps):
    """
    Calculate the parcentage of steps taken from the previous step to the next step
    (transition rate)
    Args:
        logs (list): Funnel logs. Each log is a dictionary containing user_id and step
        steps (list): Ordered list of funnel step names
    Return:
        float: Transition rate the next step
    """
    result = []
    for i in range(len(steps) - 1):
        current_step = steps[i]
        next_step = steps[i +1]

        current_count = len(get_users_at_step(logs, current_step))
        next_count = len(get_users_at_step(logs, next_step))

        rate = next_count / current_count * 100
        result.append({"from": current_step, "to": next_step, "rate": rate})

    return result

def print_funnel_summary(logs, steps):
    """
    Print the number of users, transition rate, and drop-off rate for each funnel step.
    Args:
        logs (list): Funnel logs. Each log is a dictionary containing user_id and step.
        stes (list): Orderd list of funnel step names.
    """
    for i, step in enumerate(steps):
        count = len(get_users_at_step(logs, step))

        if i  < len(steps) - 1:
            next_count = len(get_users_at_step(logs, steps[i + 1]))
            transition_rate = next_count / count * 100
            drop_rate = 100 - transition_rate
        else:
            transition_rate = None
            drop_rate = None
        
        if transition_rate is None:
            print(f"{step:<15}: {count:>2} users  (last step)")
        else:
            print(f"{step:<15}: {count:>2} users → transition: {transition_rate:5.1f}% drop-off: {drop_rate:5.1f}%")


print_funnel_summary(funnel_logs, FUNNEL_STEPS)


top_page       :  5 users → transition:  80.0% drop-off:  20.0%
product_page   :  4 users → transition:  50.0% drop-off:  50.0%
cart           :  2 users → transition:  50.0% drop-off:  50.0%
purchase       :  1 users  (last step)
